# 29 — Federated Learning Fundamentals — Lab

This lab simulates a federated learning system using the Flower framework. You will implement a FedAvg simulation on MNIST and observe how label skew (non-IID data distribution) degrades convergence compared to IID training. The lab reinforces the FedAvg algorithm, client-server orchestration, and the practical impact of data heterogeneity on federated learning.

- State the FedAvg algorithm step by step, including round structure, local training, weighted averaging, and convergence criteria.
- Distinguish cross-device federated learning from cross-silo federated learning by scale, trust model, and data quantity per client.
- Implement a FedAvg simulation using Flower with configurable numbers of virtual clients, local epochs, and participation fraction.
- Explain why non-IID label skew degrades FedAvg convergence and quantify the accuracy gap versus IID training on a fixed dataset.

You should have completed the **lecture 29** and understand the FedAvg algorithm, the FedAvg pseudocode, cross-device and cross-silo FL settings, and the impact of non-IID data on convergence. Familiarity with training neural networks on MNIST and basic gradient descent hyperparameters is assumed.

Required software/packages: flower (flwr) >= 1.5.0, tensorflow >= 2.12.0, numpy >= 1.23.0, matplotlib >= 3.7.0, scipy.

Run the following cell to install and import required packages.

**What this cell does:** Makes sure the Flower framework (`flwr`) is available. It tries to import it and, if missing, installs it with pip — including the `[simulation]` extra, which provides the virtual-client simulation engine used later in the lab (the `<2.0` pin keeps the API used here stable). If Flower is already installed, the cell does nothing.

In [ ]:
import subprocess
import sys

# Install Flower if not present
try:
    import flwr
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "flwr[simulation]>=1.5.0,<2.0"])
    import flwr
    print("Flower installed successfully.")

**What this cell does:** Imports all required libraries and fixes the random seeds so results are reproducible. It then loads MNIST, scales pixel values from [0, 255] to [0, 1], and adds a channel dimension so each image has shape (28, 28, 1) — the format the CNN expects. The full dataset stays in memory; the partitioning step later hands each simulated client its own slice of it.

In [ ]:


# Standard imports
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import flwr as fl
from scipy.stats import dirichlet

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Load MNIST dataset (28x28 grayscale, add a channel dimension)
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train[..., np.newaxis] / 255.0
x_test = x_test[..., np.newaxis] / 255.0
y_train, y_test = y_train.flatten(), y_test.flatten()
print(f"Setup complete. MNIST loaded: train shape {x_train.shape}, test shape {x_test.shape}")

Federated Learning trains a model by distributing training across multiple clients, each holding private data. The FedAvg algorithm orchestrates this: each round, the server broadcasts the global model to clients, each client trains locally for E epochs, clients upload updated weights, the server averages them and repeats. In this lab, you will simulate this flow using Flower on MNIST. You will run two experiments: IID data (all clients see all classes equally) and non-IID data (Dirichlet label skew, clients see different subsets of classes). Comparing convergence curves will show how label skew degrades federated training.

1. Implement functions to create IID and non-IID partitions of MNIST across 10 clients, and implement a Flower NumPyClient with fit() and evaluate() methods. (6 min)
2. Run the IID simulation: partition MNIST training data uniformly across 10 virtual clients, run FedAvg for 20 rounds with C=1.0 (all clients participate), record global accuracy per round. (9 min)
3. Run the Non-IID simulation: re-partition the same MNIST dataset using Dirichlet alpha=0.1 label distribution, run the same 20-round FedAvg, record global accuracy per round. (9 min)
4. Plot both accuracy curves (IID and Non-IID) on the same axes, compute and report the final-round accuracy gap between IID and Non-IID runs. (6 min)

**What this cell does:** Defines the two data-partitioning strategies compared in this lab. `create_iid_partitions` shuffles all sample indices and splits them uniformly, so every client receives an equal-sized, class-balanced slice of MNIST. `create_noniid_partitions` splits each class separately using proportions drawn from a Dirichlet($\alpha$) distribution: with $\alpha=0.1$, most of each class lands on only a few clients, so every client ends up with a different, skewed label mix (label skew). Both functions return one `(x_train, y_train, x_test, y_test)` tuple per client — the client's private data that never leaves the client.

In [ ]:
# IID and non-IID partitioning functions

def create_iid_partitions(x_train: np.ndarray, y_train: np.ndarray, x_test: np.ndarray, y_test: np.ndarray, num_clients: int) -> List[Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]]:
    """
    Partition MNIST training and test data uniformly across clients (IID).
    Returns a list of (x_train_client, y_train_client, x_test_client, y_test_client) tuples.
    """
    # Shuffle indices, then split uniformly so each client sees all 10 classes in similar proportions
    train_idx = np.random.permutation(len(x_train))
    test_idx = np.random.permutation(len(x_test))
    train_splits = np.array_split(train_idx, num_clients)
    test_splits = np.array_split(test_idx, num_clients)
    return [
        (x_train[tr], y_train[tr], x_test[te], y_test[te])
        for tr, te in zip(train_splits, test_splits)
    ]

def create_noniid_partitions(x_train: np.ndarray, y_train: np.ndarray, x_test: np.ndarray, y_test: np.ndarray, num_clients: int, alpha: float) -> List[Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]]:
    """
    Partition MNIST data using Dirichlet label distribution (non-IID).
    Each client samples a label distribution from Dirichlet(alpha) for each class.
    Returns a list of (x_train_client, y_train_client, x_test_client, y_test_client) tuples.
    """
    def dirichlet_split(y: np.ndarray) -> List[np.ndarray]:
        # For each class, draw per-client proportions from Dirichlet(alpha) and split that class's indices accordingly
        client_indices: List[List[int]] = [[] for _ in range(num_clients)]
        for c in range(10):
            class_idx = np.where(y == c)[0]
            np.random.shuffle(class_idx)
            proportions = np.random.dirichlet([alpha] * num_clients)
            split_points = (np.cumsum(proportions) * len(class_idx)).astype(int)[:-1]
            for k, split in enumerate(np.split(class_idx, split_points)):
                client_indices[k].extend(split.tolist())
        return [np.array(idx, dtype=int) for idx in client_indices]

    train_client_idx = dirichlet_split(y_train)
    test_client_idx = dirichlet_split(y_test)
    return [
        (x_train[train_client_idx[k]], y_train[train_client_idx[k]],
         x_test[test_client_idx[k]], y_test[test_client_idx[k]])
        for k in range(num_clients)
    ]

print("Partitioning functions defined.")

**What this cell does:** Defines the model and the Flower client. `create_model` builds a small CNN for MNIST — deliberately tiny so that many rounds of federated training run quickly on a laptop. `FedAvgClient` wraps that model in Flower's `NumPyClient` interface: `get_parameters` returns the current weights as NumPy arrays; `fit` installs the global weights received from the server, trains locally for `local_epochs` epochs with the configured `batch_size`, and returns the updated weights together with the number of local training samples (FedAvg uses that count for the weighted average); `evaluate` installs the global weights and reports loss and accuracy on the client's local test split.

In [ ]:
# Define model and Flower client

def create_model() -> keras.Model:
    """
    Create a small CNN model for MNIST.
    """
    model = keras.Sequential([
        layers.Conv2D(16, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(32, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

class FedAvgClient(fl.client.NumPyClient):
    """
    Flower NumPyClient for federated learning on MNIST.
    """
    def __init__(self, x_train: np.ndarray, y_train: np.ndarray, x_test: np.ndarray, y_test: np.ndarray):
        self.x_train = x_train
        self.y_train = y_train
        self.x_test = x_test
        self.y_test = y_test
        self.model = create_model()
    
    def get_parameters(self, config: Dict) -> List[np.ndarray]:
        return self.model.get_weights()
    
    def fit(self, parameters: List[np.ndarray], config: Dict) -> Tuple[List[np.ndarray], int, Dict]:
        """
        Set model parameters, train locally for E epochs with configurable batch size, return updated parameters.
        """
        self.model.set_weights(parameters)
        local_epochs = config.get('local_epochs', 2)
        batch_size = config.get('batch_size', 32)
        self.model.fit(self.x_train, self.y_train, epochs=local_epochs, batch_size=batch_size, verbose=0)
        return self.model.get_weights(), len(self.x_train), {}
    
    def evaluate(self, parameters: List[np.ndarray], config: Dict) -> Tuple[float, int, Dict]:
        """
        Set model parameters, evaluate on local test data, return loss and accuracy.
        """
        self.model.set_weights(parameters)
        loss, accuracy = self.model.evaluate(self.x_test, self.y_test, verbose=0)
        return loss, len(self.x_test), {"accuracy": accuracy}

print("Model and Flower client defined.")

**What this cell does:** Runs the two experiments. `run_fedavg_simulation` is the engine: `client_fn` creates one virtual client per data partition when Flower asks for it; the `FedAvg` strategy performs the weighted averaging of client updates, with `fraction_fit=1.0` meaning $C=1.0$ (all clients participate every round); `on_fit_config_fn` ships the local-epoch and batch-size settings to clients each round; and `evaluate_fn` runs on the server after each round, evaluating the aggregated global model on the full test set and recording one accuracy per round (the round-0 evaluation before any training is skipped). The cell runs the identical 20-round simulation twice — once on IID partitions, once on Dirichlet($\alpha=0.1$) non-IID partitions — and stores the per-round accuracies for comparison.

In [ ]:
# IID and Non-IID simulations with Flower

NUM_CLIENTS = 10
NUM_ROUNDS = 20
LOCAL_EPOCHS = 2
BATCH_SIZE = 32

def run_fedavg_simulation(partitions: List[Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]],
                          num_rounds: int, local_epochs: int, batch_size: int) -> List[float]:
    """
    Run a FedAvg simulation over the given client partitions.
    Returns a list of global test accuracies, one per round.
    """
    def client_fn(cid: str) -> fl.client.Client:
        x_tr, y_tr, x_te, y_te = partitions[int(cid)]
        return FedAvgClient(x_tr, y_tr, x_te, y_te).to_client()

    # Server-side evaluation of the global model on the full test set after each round
    global_model = create_model()
    accuracies: List[float] = []

    def evaluate_fn(server_round: int, parameters, config):
        # Recent flwr 1.x passes a list of ndarrays; older releases pass a Parameters proto
        weights = fl.common.parameters_to_ndarrays(parameters) if hasattr(parameters, "tensors") else parameters
        global_model.set_weights(weights)
        loss, accuracy = global_model.evaluate(x_test, y_test, verbose=0)
        if server_round > 0:  # skip the initial pre-training evaluation (round 0)
            accuracies.append(accuracy)
            print(f"Round {server_round}: global accuracy = {accuracy:.4f}")
        return loss, {"accuracy": accuracy}

    strategy = fl.server.strategy.FedAvg(
        fraction_fit=1.0,       # C = 1.0: all clients participate
        fraction_evaluate=1.0,
        min_fit_clients=len(partitions),
        min_evaluate_clients=len(partitions),
        min_available_clients=len(partitions),
        evaluate_fn=evaluate_fn,
        on_fit_config_fn=lambda rnd: {"local_epochs": local_epochs, "batch_size": batch_size},
    )

    fl.simulation.start_simulation(
        client_fn=client_fn,
        num_clients=len(partitions),
        config=fl.server.ServerConfig(num_rounds=num_rounds),
        strategy=strategy,
    )
    return accuracies

# IID simulation: uniform partitions, 20 rounds, C=1.0, batch_size=32
print("Starting IID simulation...")
iid_partitions = create_iid_partitions(x_train, y_train, x_test, y_test, NUM_CLIENTS)
iid_accuracies = run_fedavg_simulation(iid_partitions, NUM_ROUNDS, LOCAL_EPOCHS, BATCH_SIZE)

# Non-IID simulation: Dirichlet(alpha=0.1) label skew, 20 rounds, C=1.0, batch_size=32
print("Starting non-IID simulation...")
noniid_partitions = create_noniid_partitions(x_train, y_train, x_test, y_test, NUM_CLIENTS, alpha=0.1)
noniid_accuracies = run_fedavg_simulation(noniid_partitions, NUM_ROUNDS, LOCAL_EPOCHS, BATCH_SIZE)

**What this cell does:** Visualizes the results. It plots the IID and non-IID accuracy curves on the same axes (one point per round) and computes the final-round accuracy gap, IID minus non-IID. Expect the IID curve to sit higher and converge smoothly, while the non-IID curve stays lower and oscillates more — the visible signature of client drift caused by label skew.

In [ ]:
# Plot both accuracy curves and compute the accuracy gap

plt.figure(figsize=(10, 6))
rounds = range(1, len(iid_accuracies) + 1)
plt.plot(rounds, iid_accuracies, marker='o', label='IID')
plt.plot(rounds, noniid_accuracies, marker='s', label='Non-IID (Dirichlet $\alpha=0.1$)')
plt.xlabel('Round')
plt.ylabel('Global Accuracy')
plt.title('IID vs Non-IID Federated Learning Convergence')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Compute and print accuracy gap
gap = iid_accuracies[-1] - noniid_accuracies[-1]
print(f"Accuracy gap (IID - Non-IID): {gap:.4f}")

Expected output: A plot showing two accuracy curves, one for IID (higher accuracy, smoother convergence) and one for non-IID (lower accuracy, more oscillation). The final-round IID accuracy should be approximately 0.97-0.99, and the non-IID accuracy approximately 0.85-0.95, resulting in a gap of roughly 0.05-0.15.

Comprehension questions:

1. Why does the non-IID accuracy curve show more oscillation than the IID curve? What does this oscillation tell you about the aggregation process?
2. In your non-IID simulation, each client sees only a subset of MNIST classes due to Dirichlet(alpha=0.1) label skew. How does this affect the local training loss versus the global test accuracy?

**Answer 1:** Under non-IID label skew, each client's local update pulls the model toward the few classes it owns, so successive rounds are dominated by different class mixtures. The weighted average keeps shifting direction instead of making steady progress, which shows up as oscillation in the accuracy curve. The oscillation is a symptom of the server averaging divergent, mutually incompatible local models: the aggregate is a weak compromise that alternately helps and hurts different classes. IID clients share the same distribution, so their updates are aligned and the curve converges smoothly.

**Answer 2:** Local training loss on a skewed client drops quickly, because fitting 2-3 classes is an easier problem than fitting all 10. The client over-specializes to its own label subset. But that specialization does not generalize: classes the client never saw are effectively forgotten (client drift), so the global model's accuracy on the full 10-class test set lags well behind what the low local losses suggest. Low local loss combined with poor global accuracy is the signature of client drift caused by label skew.

### Assignment:

Modify the simulation settings and re-run the notebook to explore how FedAvg's hyperparameters affect convergence. Complete all four tasks, fill in the results table, and answer the questions below.

**Baseline (what the notebook runs above):** `NUM_CLIENTS=10`, `NUM_ROUNDS=20`, `LOCAL_EPOCHS=2`, `BATCH_SIZE=32`, `fraction_fit=1.0` (all clients participate, $C=1.0$), and Dirichlet skew `alpha=0.1` for the non-IID run.

Tip: when a task says "non-IID only", you can comment out the IID simulation call and reuse your baseline IID numbers — the IID run does not change. Always restore the baseline value of a setting before moving to the next task.

#### Task 1 — More local training ($E$: 2 → 5)
Change `LOCAL_EPOCHS = 5` in the simulation cell and re-run **both** simulations.

*Question:* Does the IID/non-IID accuracy gap shrink or grow compared to baseline? Explain in terms of client drift and communication cost.

#### Task 2 — Partial participation ($C$: 1.0 → 0.5)
Restore `LOCAL_EPOCHS = 2`. In the `FedAvg` strategy, change `fraction_fit=0.5` and `min_fit_clients=5` (only 5 of the 10 clients are now sampled for training each round; leave the other settings unchanged). Re-run **both** simulations.

*Question:* How does convergence per round compare to the baseline? If each client's model upload were 5 MB, how much upload bandwidth would this save per round?

#### Task 3 — Weaker label skew ($\alpha$: 0.1 → 1.0)
Restore `fraction_fit=1.0` and `min_fit_clients=10`. In the non-IID partition call, change `alpha=0.1` to `alpha=1.0`, then re-run the **non-IID simulation only**.

*Question:* How does the gap change as client data becomes more IID-like? Why?

#### Task 4 — Larger batches (batch size: 32 → 128)
Restore `alpha=0.1`. Change `BATCH_SIZE = 128` and re-run **both** simulations. Recall that each epoch performs $\tau = \lceil n_k / b \rceil$ gradient updates.

*Question:* With 6000 local training samples per client, how many gradient updates per round does each client perform now versus the baseline? How does this affect convergence per round and the smoothness of the curves?

#### Results table (fill in the final-round accuracies)

| Configuration | IID final acc | Non-IID final acc | Gap |
|---|---|---|---|
| Baseline ($E$=2, $C$=1.0, $\alpha$=0.1, $b$=32) | | | |
| Task 1 ($E$=5) | | | |
| Task 2 ($C$=0.5) | | | |
| Task 3 ($\alpha$=1.0) | — | | |
| Task 4 ($b$=128) | | | |

#### Written questions
1. Which single change reduced the accuracy gap the most? Explain why, using the ideas of client drift and model oscillation.
2. You are deploying a keyboard-prediction app on millions of smartphones (cross-device FL). Which combination of $E$, $C$, and batch size would you choose, and why?
3. How would your answer change for a consortium of 10 hospitals (cross-silo FL)?

**Deliverables:** this notebook with all cells executed top to bottom, the completed results table, and your answers to the four task questions plus the three written questions above.